# Model Improvement and Hyperparameter Tuning

This notebook focuses on improving the baseline sales forecasting model developed in the original forecasting notebook.

The improvement process includes:

- Recreating the processed modeling dataset
- Creating a chronological validation split
- Adding improved time-series features
- Training tuned machine learning models
- Comparing the improved models with the baseline Random Forest
- Evaluating the final model on the untouched test period

In [2]:


import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error


## Load Processed Dataset

Load the processed daily sales dataset created in the baseline notebook.

In [3]:
daily_sales = pd.read_csv(
    "../data/processed/daily_sales_model_data.csv",
    parse_dates=["date"]
)

daily_sales = daily_sales.sort_values("date").reset_index(drop=True)

print("Dataset shape:", daily_sales.shape)
display(daily_sales.head())

Dataset shape: (1654, 16)


,date,total_sales,day_of_week,month,year,quarter,day,day_of_week_num,is_weekend,lag_1,lag_7,lag_14,lag_30,rolling_mean_7,rolling_mean_14,rolling_mean_30
0,2013-01-31,271254.217996,Thursday,1,2013,1,31,3,0,281061.127052,247245.690995,267498.515975,2511.618999,316070.048174,325739.742089,335212.350630
1,2013-02-01,369402.055266,Friday,2,2013,1,1,4,0,271254.217996,290022.771930,296130.850028,496092.417944,319499.837745,326008.006519,344170.437264
2,2013-02-02,518887.462705,Saturday,2,2013,1,2,5,1,369402.055266,413799.767975,432459.852021,361461.231124,330839.735365,331241.664036,339947.425174
3,2013-02-03,486336.820180,Sunday,2,2013,1,3,6,1,518887.462705,430411.991233,461092.386047,354459.677093,345852.263183,337415.064800,345194.966227
4,2013-02-04,344308.715017,Monday,2,2013,1,4,0,0,486336.820180,285460.169953,311211.265950,477350.121229,353841.524461,339218.238666,349590.870997


## Chronological Validation Split

Separate the historical data into training and validation periods.

A chronological split is important for forecasting because future observations must not be used to train the model.

In [4]:
split_date = "2016-07-01"

train_data = daily_sales[daily_sales["date"] < split_date].copy()
validation_data = daily_sales[daily_sales["date"] >= split_date].copy()

print("Training period:", train_data["date"].min(), "to", train_data["date"].max())
print("Validation period:", validation_data["date"].min(), "to", validation_data["date"].max())
print("Training rows:", len(train_data))
print("Validation rows:", len(validation_data))

Training period: 2013-01-31 00:00:00 to 2016-06-30 00:00:00
Validation period: 2016-07-01 00:00:00 to 2017-08-15 00:00:00
Training rows: 1244
Validation rows: 410


## Improved Time-Series Features

In [5]:
data = daily_sales.copy()

data["lag_3"] = data["total_sales"].shift(3)
data["lag_5"] = data["total_sales"].shift(5)
data["lag_21"] = data["total_sales"].shift(21)
data["lag_28"] = data["total_sales"].shift(28)

# Shift first to prevent future information from entering the features.
data["rolling_mean_3"] = data["total_sales"].shift(1).rolling(3).mean()
data["rolling_mean_7"] = data["total_sales"].shift(1).rolling(7).mean()
data["rolling_mean_14"] = data["total_sales"].shift(1).rolling(14).mean()
data["rolling_mean_30"] = data["total_sales"].shift(1).rolling(30).mean()

data["rolling_std_7"] = data["total_sales"].shift(1).rolling(7).std()
data["rolling_std_14"] = data["total_sales"].shift(1).rolling(14).std()

data["ewm_7"] = data["total_sales"].shift(1).ewm(span=7).mean()
data["ewm_14"] = data["total_sales"].shift(1).ewm(span=14).mean()

data = data.dropna().reset_index(drop=True)

print("Improved dataset shape:", data.shape)
display(data.head())

Improved dataset shape: (1624, 25)


,date,total_sales,day_of_week,month,year,quarter,day,day_of_week_num,is_weekend,lag_1,...,rolling_mean_30,lag_3,lag_5,lag_21,lag_28,rolling_mean_3,rolling_std_7,rolling_std_14,ewm_7,ewm_14
0,2013-03-02,547857.205258,Saturday,3,2013,1,2,5,1,378898.165064,...,343637.072017,288401.998919,307110.875933,416176.279075,518887.462705,315138.818624,72958.989689,72216.486404,327177.120816,330907.871116
1,2013-03-03,559820.580027,Sunday,3,2013,1,3,6,1,547857.205258,...,352857.171592,278116.291888,270425.007067,300951.328993,486336.820180,401623.887403,105590.382324,82873.157984,382354.532200,360181.094398
2,2013-03-04,383647.679097,Monday,3,2013,1,4,0,0,559820.580027,...,359204.455751,378898.165064,288401.998919,278627.588926,344308.715017,495525.316783,126806.582752,100082.539216,426725.501328,387075.707225
3,2013-03-05,332189.993189,Tuesday,3,2013,1,5,1,0,383647.679097,...,354696.462964,547857.205258,278116.291888,308800.790912,321245.839130,497108.488127,123143.106326,100292.701391,415955.234346,386614.534934
4,2013-03-06,335726.580968,Wednesday,3,2013,1,6,2,0,332189.993189,...,349558.235397,559820.580027,378898.165064,319597.929962,321538.177149,425219.417438,115388.558450,98847.743098,395012.740712,379301.557111


## Improved Feature Set

In [6]:
features_improved = [
    "year",
    "month",
    "quarter",
    "day",
    "day_of_week_num",
    "is_weekend",
    "lag_1",
    "lag_3",
    "lag_5",
    "lag_7",
    "lag_14",
    "lag_21",
    "lag_28",
    "lag_30",
    "rolling_mean_3",
    "rolling_mean_7",
    "rolling_mean_14",
    "rolling_mean_30",
    "rolling_std_7",
    "rolling_std_14",
    "ewm_7",
    "ewm_14"
]

X = data[features_improved]
y = data["total_sales"]

train_data = data[data["date"] < split_date]
validation_data = data[data["date"] >= split_date]

X_train = train_data[features_improved]
y_train = train_data["total_sales"]

X_validation = validation_data[features_improved]
y_validation = validation_data["total_sales"]

print("Training shape:", X_train.shape)
print("Validation shape:", X_validation.shape)

Training shape: (1214, 22)
Validation shape: (410, 22)


## Improved Random Forest

In [7]:
rf_improved = RandomForestRegressor(
    n_estimators=200,
    max_depth=20,
    min_samples_split=5,
    min_samples_leaf=2,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1
)

rf_improved.fit(X_train, y_train)

validation_predictions = rf_improved.predict(X_validation)

print("Improved Random Forest trained successfully.")

Improved Random Forest trained successfully.


## Validation Performance

In [8]:
mae_improved = mean_absolute_error(
    y_validation,
    validation_predictions
)

rmse_improved = np.sqrt(
    mean_squared_error(
        y_validation,
        validation_predictions
    )
)

mape_improved = np.mean(
    np.abs(
        (y_validation - validation_predictions) /
        np.where(y_validation == 0, 1, y_validation)
    )
) * 100

print("Improved Random Forest")
print("MAE:", round(mae_improved, 2))
print("RMSE:", round(rmse_improved, 2))
print("MAPE:", round(mape_improved, 2), "%")

Improved Random Forest
MAE: 70350.96
RMSE: 110149.09
MAPE: 28.28 %


## Model Comparison

In [9]:
comparison = pd.DataFrame({
    "Model": [
        "Baseline Random Forest",
        "Improved Random Forest"
    ],
    "MAE": [
        76821.42,
        mae_improved
    ],
    "RMSE": [
        133556.96,
        rmse_improved
    ],
    "MAPE": [
        47.50,
        mape_improved
    ]
})

display(comparison.round(2))

,Model,MAE,RMSE,MAPE
0,Baseline Random Forest,76821.42,133556.96,47.50
1,Improved Random Forest,70350.96,110149.09,28.28


## Random Forest Hyperparameter Tuning

In [10]:
from sklearn.model_selection import RandomizedSearchCV

param_grid = {
    "n_estimators": [200, 300, 400],
    "max_depth": [15, 20, 25, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", 0.7, 1.0]
}

rf_search = RandomizedSearchCV(
    RandomForestRegressor(random_state=42, n_jobs=-1),
    param_distributions=param_grid,
    n_iter=15,
    scoring="neg_mean_absolute_error",
    cv=3,
    random_state=42,
    n_jobs=-1
)

rf_search.fit(X_train, y_train)

print("Best parameters:")
print(rf_search.best_params_)
print("Best CV MAE:", round(-rf_search.best_score_, 2))

Best parameters:
{'n_estimators': 200, 'min_samples_split': 2, 'min_samples_leaf': 4, 'max_features': 0.7, 'max_depth': 20}
Best CV MAE: 65381.05


## Tuned Model Validation

In [11]:
tuned_rf = rf_search.best_estimator_

tuned_predictions = tuned_rf.predict(X_validation)

mae_tuned = mean_absolute_error(
    y_validation,
    tuned_predictions
)

rmse_tuned = np.sqrt(
    mean_squared_error(
        y_validation,
        tuned_predictions
    )
)

mape_tuned = np.mean(
    np.abs(
        (y_validation - tuned_predictions) /
        np.where(y_validation == 0, 1, y_validation)
    )
) * 100

print("Tuned Random Forest")
print("MAE:", round(mae_tuned, 2))
print("RMSE:", round(rmse_tuned, 2))
print("MAPE:", round(mape_tuned, 2), "%")

Tuned Random Forest
MAE: 70410.6
RMSE: 110480.48
MAPE: 29.45 %
